In [ ]:
import pandas as pd
import os

output_dir = "../../Clean-Data-Files"
chunksize  = 100000

os.makedirs(output_dir, exist_ok=True)

print("=" * 60)
print(" BUILDING CLEAN COMPANIES DATASET ")
print("=" * 60)

# ============================================================
# STEP 1: LOAD VALID PATENT IDs FROM clean_patents.csv
# ============================================================
print("\n Loading valid patent IDs from clean_patents.csv...")

clean_patents    = pd.read_csv(f"{output_dir}/clean_patents.csv", dtype=str)
valid_patent_ids = set(clean_patents["patent_id"].str.strip())
print(f" Valid patent IDs: {len(valid_patent_ids):,}")

# ============================================================
# STEP 2: PROCESS COMPANIES FILTERED BY VALID PATENT IDs
# ============================================================
print("\n Processing companies...")

seen_ids   = set()
total      = 0
asg_chunks = []

for chunk in pd.read_csv("../../Data-Source/g_assignee_disambiguated.tsv",
                          sep="\t", dtype=str, chunksize=chunksize, on_bad_lines="skip"):

    # Filter by valid patent IDs and organizations only
    chunk["patent_id"]   = chunk["patent_id"].str.strip()
    chunk["assignee_id"] = chunk["assignee_id"].str.strip()
    chunk = chunk[
        chunk["patent_id"].isin(valid_patent_ids) &
        chunk["disambig_assignee_organization"].notna()
    ].copy()

    if chunk.empty:
        continue

    # Clean
    chunk["company_id"] = chunk["assignee_id"]
    chunk["name"]       = chunk["disambig_assignee_organization"].str.strip() \
                               .str.replace(r'\s+', ' ', regex=True)

    # Drop empty names
    chunk = chunk[chunk["name"] != ""]

    # Keep only needed columns
    chunk = chunk[["company_id", "name"]]

    # Remove duplicates across chunks
    chunk = chunk.drop_duplicates(subset=["company_id"])
    chunk = chunk[~chunk["company_id"].isin(seen_ids)]

    if chunk.empty:
        continue

    seen_ids.update(chunk["company_id"].tolist())
    asg_chunks.append(chunk)
    total += len(chunk)
    print(f"   {len(chunk)} companies saved (Total: {total:,})")

# ============================================================
# STEP 3: SAVE
# ============================================================
companies = pd.concat(asg_chunks, ignore_index=True)
companies.to_csv(f"{output_dir}/clean_companies.csv", index=False)

print("\n" + "=" * 60)
print(" VERIFICATION")
print("=" * 60)
print(f"   Total rows     : {len(companies):,}")
print(f"   Unique IDs     : {companies['company_id'].nunique():,}")
print(f"   Unique names   : {companies['name'].nunique():,}")
print(f"   Missing IDs    : {companies['company_id'].isna().sum()}")
print(f"   Missing names  : {companies['name'].isna().sum()}")
print("\n Sample:")
print(companies.head())
print("\n clean_companies.csv rebuilt successfully!")

🏢 REBUILDING clean_companies.csv

📥 Loading valid patent IDs from clean_patents.csv...
✅ Valid patent IDs: 8,434,027

📥 Processing companies...
   ✅ 31085 companies saved (Total: 31,085)
   ✅ 19902 companies saved (Total: 50,987)
   ✅ 16826 companies saved (Total: 67,813)
   ✅ 14843 companies saved (Total: 82,656)
   ✅ 13252 companies saved (Total: 95,908)
   ✅ 12541 companies saved (Total: 108,449)
   ✅ 11479 companies saved (Total: 119,928)
   ✅ 11024 companies saved (Total: 130,952)
   ✅ 10357 companies saved (Total: 141,309)
   ✅ 9619 companies saved (Total: 150,928)
   ✅ 9468 companies saved (Total: 160,396)
   ✅ 8841 companies saved (Total: 169,237)
   ✅ 8763 companies saved (Total: 178,000)
   ✅ 8456 companies saved (Total: 186,456)
   ✅ 8016 companies saved (Total: 194,472)
   ✅ 7862 companies saved (Total: 202,334)
   ✅ 7444 companies saved (Total: 209,778)
   ✅ 7343 companies saved (Total: 217,121)
   ✅ 7230 companies saved (Total: 224,351)
   ✅ 6758 companies saved (Total: 2

In [ ]:
import pandas as pd
import os

output_dir = "../../Clean-Data-Files"
chunksize  = 100000

os.makedirs(output_dir, exist_ok=True)

print("=" * 60)
print(" BUILDING CLEAN INVENTORS DATASET ")
print("=" * 60)

# ============================================================
# STEP 1: LOAD VALID PATENT IDs FROM clean_patents.csv
# ============================================================
print("\n Loading valid patent IDs from clean_patents.csv...")

clean_patents    = pd.read_csv(f"{output_dir}/clean_patents.csv", dtype=str)
valid_patent_ids = set(clean_patents["patent_id"].str.strip())
print(f" Valid patent IDs: {len(valid_patent_ids):,}")

# ============================================================
# STEP 2: LOAD LOCATIONS
# ============================================================
print("\n Loading locations...")

location = pd.read_csv("../../Data-Source/g_location_disambiguated.tsv",
                        sep="\t", dtype=str,
                        usecols=["location_id", "disambig_country"],
                        on_bad_lines="skip")
location["location_id"]      = location["location_id"].str.strip()
location["disambig_country"] = location["disambig_country"].str.strip()
location = location.dropna()
print(f" Valid locations: {len(location):,}")

# ============================================================
# STEP 3: PROCESS INVENTORS FILTERED BY VALID PATENT IDs
# ============================================================
print("\n Processing inventors...")

seen_ids   = set()
total      = 0
inv_chunks = []

for chunk in pd.read_csv("../../Data-Source/g_inventor_disambiguated.tsv",
                          sep="\t", dtype=str, chunksize=chunksize, on_bad_lines="skip"):

    # Filter by valid patent IDs
    chunk["patent_id"]   = chunk["patent_id"].str.strip()
    chunk["inventor_id"] = chunk["inventor_id"].str.strip()
    chunk = chunk[chunk["patent_id"].isin(valid_patent_ids)].copy()

    if chunk.empty:
        continue

    # Merge location for country
    chunk = chunk.merge(location, on="location_id", how="left")

    # Build full name
    chunk["name"] = (
        chunk["disambig_inventor_name_first"].fillna("") + " " +
        chunk["disambig_inventor_name_last"].fillna("")
    ).str.strip().str.replace(r'\s+', ' ', regex=True).str.title()
    chunk["name"] = chunk["name"].replace("", "Unknown")

    # Clean country
    chunk["country"] = chunk["disambig_country"].fillna("Unknown").str.strip().str.title()

    # Keep only needed columns
    chunk = chunk[["inventor_id", "name", "country"]]

    # Remove duplicates across chunks
    chunk = chunk.drop_duplicates(subset=["inventor_id"])
    chunk = chunk[~chunk["inventor_id"].isin(seen_ids)]

    if chunk.empty:
        continue

    seen_ids.update(chunk["inventor_id"].tolist())
    inv_chunks.append(chunk)
    total += len(chunk)
    print(f"   {len(chunk)} inventors saved (Total: {total:,})")

# ============================================================
# STEP 4: SAVE
# ============================================================
inventors = pd.concat(inv_chunks, ignore_index=True)
inventors.to_csv(f"{output_dir}/clean_inventors.csv", index=False)

print("\n" + "=" * 60)
print(" VERIFICATION")
print("=" * 60)
print(f"   Total rows       : {len(inventors):,}")
print(f"   Unique IDs       : {inventors['inventor_id'].nunique():,}")
print(f"   Unique names     : {inventors['name'].nunique():,}")
print(f"   Unique countries : {inventors['country'].nunique():,}")
print(f"   Missing names    : {inventors['name'].isna().sum()}")
print(f"   Missing countries: {inventors['country'].isna().sum()}")
print("\n Top Countries:")
print(inventors["country"].value_counts().head(5))
print("\n Sample:")
print(inventors.head())
print("\n clean_inventors.csv rebuilt successfully!")

👥 REBUILDING clean_inventors.csv

📥 Loading valid patent IDs from clean_patents.csv...
✅ Valid patent IDs: 8,434,027

📥 Loading locations...
✅ Locations loaded: 100,442

📥 Processing inventors...
   ✅ 87039 inventors saved (Total: 87,039)
   ✅ 77316 inventors saved (Total: 164,355)
   ✅ 70466 inventors saved (Total: 234,821)
   ✅ 65540 inventors saved (Total: 300,361)
   ✅ 60913 inventors saved (Total: 361,274)
   ✅ 57894 inventors saved (Total: 419,168)
   ✅ 54934 inventors saved (Total: 474,102)
   ✅ 51858 inventors saved (Total: 525,960)
   ✅ 49679 inventors saved (Total: 575,639)
   ✅ 47558 inventors saved (Total: 623,197)
   ✅ 46131 inventors saved (Total: 669,328)
   ✅ 44243 inventors saved (Total: 713,571)
   ✅ 42608 inventors saved (Total: 756,179)
   ✅ 41252 inventors saved (Total: 797,431)
   ✅ 40108 inventors saved (Total: 837,539)
   ✅ 38579 inventors saved (Total: 876,118)
   ✅ 37804 inventors saved (Total: 913,922)
   ✅ 36632 inventors saved (Total: 950,554)
   ✅ 35837 in

In [ ]:
import pandas as pd
import os

output_dir = "../../Clean-Data-Files"
chunksize  = 100000

os.makedirs(output_dir, exist_ok=True)

# ============================================================
# STEP 1: COLLECT COMMON PATENT IDs
# ============================================================
print(" Collecting patent IDs from inventor file...")
inv_patent_ids = set()
for chunk in pd.read_csv("../../Data-Source/g_inventor_disambiguated.tsv",
                          sep="\t", dtype=str, chunksize=chunksize, on_bad_lines="skip"):
    inv_patent_ids.update(chunk["patent_id"].str.strip())

print(f" Inventor patent IDs: {len(inv_patent_ids):,}")

print("\n Collecting patent IDs from assignee file...")
asg_patent_ids = set()
for chunk in pd.read_csv("../../Data-Source/g_assignee_disambiguated.tsv",
                          sep="\t", dtype=str, chunksize=chunksize, on_bad_lines="skip"):
    asg_patent_ids.update(chunk["patent_id"].str.strip())

print(f" Assignee patent IDs: {len(asg_patent_ids):,}")

common_ids = inv_patent_ids & asg_patent_ids
print(f" Common patent IDs: {len(common_ids):,}")

# ============================================================
# STEP 2: REBUILD clean_patents.csv
# ============================================================
print("\n📥 Rebuilding clean_patents.csv...")

pat_chunks = []
total = 0
for chunk in pd.read_csv("../../Data-Source/g_patent.tsv", sep="\t", dtype=str,
                          chunksize=chunksize, on_bad_lines="skip"):
    chunk["patent_id"] = chunk["patent_id"].str.strip()
    chunk = chunk[chunk["patent_id"].isin(common_ids)].drop_duplicates(subset=["patent_id"])
    if not chunk.empty:
        pat_chunks.append(chunk)
        total += len(chunk)
        print(f"   {len(chunk)} patents (Total: {total:,})")

patents    = pd.concat(pat_chunks, ignore_index=True)
patent_ids = set(patents["patent_id"])
print(f" Patents loaded: {len(patents):,}")

# ============================================================
# STEP 3: MERGE ABSTRACTS
# ============================================================
print("\n Loading abstracts...")
ab_chunks = []
for chunk in pd.read_csv("../../Data-Source/g_patent_abstract.tsv", sep="\t", dtype=str,
                          usecols=["patent_id", "patent_abstract"],
                          chunksize=chunksize, on_bad_lines="skip"):
    chunk = chunk[chunk["patent_id"].str.strip().isin(patent_ids)]
    if not chunk.empty:
        ab_chunks.append(chunk)

abstracts = pd.concat(ab_chunks, ignore_index=True).drop_duplicates(subset=["patent_id"])
print(f" Abstracts: {len(abstracts):,}")

# ============================================================
# STEP 4: MERGE FILING DATES
# ============================================================
print("\n Loading filing dates...")
app_chunks = []
for chunk in pd.read_csv("../../Data-Source/g_application.tsv", sep="\t", dtype=str,
                          usecols=["patent_id", "filing_date"],
                          chunksize=chunksize, on_bad_lines="skip"):
    chunk = chunk[chunk["patent_id"].str.strip().isin(patent_ids)]
    if not chunk.empty:
        app_chunks.append(chunk)

applications = pd.concat(app_chunks, ignore_index=True).drop_duplicates(subset=["patent_id"])
print(f" Filing dates: {len(applications):,}")

# ============================================================
# STEP 5: MERGE + CLEAN + SAVE
# ============================================================
print("\n Merging and cleaning...")

patents = patents.merge(abstracts,    on="patent_id", how="left")
patents = patents.merge(applications, on="patent_id", how="left")

patents["patent_title"]    = patents["patent_title"].fillna("Unknown").str.strip()
patents["patent_abstract"] = patents["patent_abstract"].fillna("No Abstract").str.strip()
patents["filing_date"]     = pd.to_datetime(patents["filing_date"], errors="coerce")
patents["year"]            = patents["filing_date"].dt.year

patents = patents[["patent_id", "patent_title", "patent_abstract", "filing_date", "year"]] \
          .rename(columns={"patent_title": "title", "patent_abstract": "abstract"}) \
          .drop_duplicates(subset=["patent_id"])

patents.to_csv(f"{output_dir}/clean_patents.csv", index=False)

# ============================================================
# VERIFY
# ============================================================
print("\n" + "=" * 60)
print(" VERIFICATION")
print("=" * 60)
print(f"   Total rows   : {len(patents):,}")
print(f"   Unique IDs   : {patents['patent_id'].nunique():,}")
print(f"   Missing dates: {patents['filing_date'].isna().sum()}")
print(f"   Missing title: {patents['title'].isna().sum()}")
print("\n Sample patent IDs:")
print(patents["patent_id"].head())
print("\n clean_patents.csv rebuilt successfully!")

📥 Collecting patent IDs from inventor file...
✅ Inventor patent IDs: 9,453,074

📥 Collecting patent IDs from assignee file...
✅ Assignee patent IDs: 8,434,880
✅ Common patent IDs: 8,434,027

📥 Rebuilding clean_patents.csv...
   ✅ 94453 patents (Total: 94,453)
   ✅ 94473 patents (Total: 188,926)
   ✅ 94385 patents (Total: 283,311)
   ✅ 94860 patents (Total: 378,171)
   ✅ 94584 patents (Total: 472,755)
   ✅ 94854 patents (Total: 567,609)
   ✅ 94638 patents (Total: 662,247)
   ✅ 94817 patents (Total: 757,064)
   ✅ 94678 patents (Total: 851,742)
   ✅ 94786 patents (Total: 946,528)
   ✅ 94909 patents (Total: 1,041,437)
   ✅ 94760 patents (Total: 1,136,197)
   ✅ 94944 patents (Total: 1,231,141)
   ✅ 94772 patents (Total: 1,325,913)
   ✅ 94818 patents (Total: 1,420,731)
   ✅ 95234 patents (Total: 1,515,965)
   ✅ 95238 patents (Total: 1,611,203)
   ✅ 95061 patents (Total: 1,706,264)
   ✅ 95259 patents (Total: 1,801,523)
   ✅ 95261 patents (Total: 1,896,784)
   ✅ 95352 patents (Total: 1,992,136

In [ ]:
import pandas as pd
import os

output_dir = "../../Clean-Data-Files"
chunksize  = 100000

print("=" * 60)
print(" BUILDING clean_relationships.csv")
print("=" * 60)

# ============================================================
# STEP 1: LOAD VALID IDs FROM CLEAN FILES
# ============================================================
print("\n Loading valid IDs from clean files...")

clean_patents   = pd.read_csv(f"{output_dir}/clean_patents.csv",   dtype=str)
clean_inventors = pd.read_csv(f"{output_dir}/clean_inventors.csv", dtype=str)
clean_companies = pd.read_csv(f"{output_dir}/clean_companies.csv", dtype=str)

valid_patent_ids   = set(clean_patents["patent_id"].str.strip())
valid_inventor_ids = set(clean_inventors["inventor_id"].str.strip())
valid_company_ids  = set(clean_companies["company_id"].str.strip())

print(f" Valid patents   : {len(valid_patent_ids):,}")
print(f" Valid inventors : {len(valid_inventor_ids):,}")
print(f" Valid companies : {len(valid_company_ids):,}")

# ============================================================
# STEP 2: BUILD PATENT-INVENTOR LINKS
# ============================================================
print("\n Loading patent-inventor links...")

pi_chunks = []
for chunk in pd.read_csv("../../Data-Source/g_inventor_disambiguated.tsv",
                          sep="\t", dtype=str, chunksize=chunksize, on_bad_lines="skip"):
    chunk["patent_id"]   = chunk["patent_id"].str.strip()
    chunk["inventor_id"] = chunk["inventor_id"].str.strip()
    chunk = chunk[
        chunk["patent_id"].isin(valid_patent_ids) &
        chunk["inventor_id"].isin(valid_inventor_ids)
    ][["patent_id", "inventor_id"]].drop_duplicates()

    if not chunk.empty:
        pi_chunks.append(chunk)

pi_df = pd.concat(pi_chunks, ignore_index=True).drop_duplicates()
print(f" Patent-inventor links: {len(pi_df):,}")

# ============================================================
# STEP 3: BUILD PATENT-COMPANY LINKS
# ============================================================
print("\n Loading patent-company links...")

pc_chunks = []
for chunk in pd.read_csv("../../Data-Source/g_assignee_disambiguated.tsv",
                          sep="\t", dtype=str, chunksize=chunksize, on_bad_lines="skip"):
    chunk["patent_id"]   = chunk["patent_id"].str.strip()
    chunk["assignee_id"] = chunk["assignee_id"].str.strip()
    chunk = chunk[
        chunk["patent_id"].isin(valid_patent_ids) &
        chunk["assignee_id"].isin(valid_company_ids)
    ][["patent_id", "assignee_id"]].rename(columns={"assignee_id": "company_id"}).drop_duplicates()

    if not chunk.empty:
        pc_chunks.append(chunk)

pc_df = pd.concat(pc_chunks, ignore_index=True).drop_duplicates()
print(f" Patent-company links: {len(pc_df):,}")

# ============================================================
# STEP 4: MERGE INTO ONE RELATIONSHIPS TABLE
# ============================================================
print("\n Merging into unified relationships table...")

relationships = pi_df.merge(pc_df, on="patent_id", how="left").drop_duplicates()
print(f" Total relationships  : {len(relationships):,}")
print(f"   With company        : {relationships['company_id'].notna().sum():,}")
print(f"   Without company     : {relationships['company_id'].isna().sum():,}")

# ============================================================
# STEP 5: SAVE
# ============================================================
relationships.to_csv(f"{output_dir}/clean_relationships.csv", index=False)

print("\n" + "=" * 60)
print(" VERIFICATION")
print("=" * 60)
print(f"   Total rows       : {len(relationships):,}")
print(f"   Unique patents   : {relationships['patent_id'].nunique():,}")
print(f"   Unique inventors : {relationships['inventor_id'].nunique():,}")
print(f"   Unique companies : {relationships['company_id'].nunique():,}")
print("\n Sample:")
print(relationships.head())
print("\n clean_relationships.csv saved successfully!")

🔗 BUILDING clean_relationships.csv

📥 Loading valid IDs from clean files...
✅ Valid patents   : 8,434,027
✅ Valid inventors : 3,867,751
✅ Valid companies : 515,987

📥 Building patent-inventor links...
✅ Patent-inventor links: 22,679,928

📥 Building patent-company links...
✅ Patent-company links: 8,660,946

🔗 Merging into unified relationships table...
✅ Total relationships  : 23,877,738
   With company        : 23,772,319
   Without company     : 105,419

✅ VERIFICATION
   Total rows       : 23,877,738
   Unique patents   : 8,434,027
   Unique inventors : 3,867,751
   Unique companies : 515,987

📋 Sample:
  patent_id           inventor_id                            company_id
0  12029253    fl:ei_ln:baumker-1  41946e18-f5c5-4724-a6ee-fc0a425d198a
1   6584128    fl:ri_ln:kroeger-1  38969e56-ae8b-409d-9dd9-f67b77acb42b
2  11161990  fl:ma_ln:boudreaux-4  92aa0a22-a0d8-4ffc-9026-aeda00ae190c
3   6795487  fl:ge_ln:whitworth-1  c75987e0-09a2-423e-ad8c-ec060cff179a
4  12324454  fl:ro_ln:egnat